In [53]:
# Install/import packages and create the AWS context

import json
import time
import boto3
import sagemaker

from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.model import Model
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.tuner import ContinuousParameter, HyperparameterTuner, IntegerParameter
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger, ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.steps import CacheConfig, ProcessingStep, TuningStep

region = boto3.Session().region_name
if not region:
    raise RuntimeError("AWS region was not detected. Open this notebook in SageMaker Studio or configure boto3.")

boto_session = boto3.Session(region_name=region)
sagemaker_session = sagemaker.Session(boto_session=boto_session)
pipeline_session = PipelineSession(boto_session=boto_session)

# This fixes the earlier NameError: name 'role' is not defined.
role = sagemaker.get_execution_role()
default_bucket = sagemaker_session.default_bucket()

print("Region:", region)
print("Execution role:", role)
print("Default bucket:", default_bucket)

Region: us-east-1
Execution role: arn:aws:iam::581187100103:role/service-role/AmazonSageMaker-ExecutionRole-20260723T215584
Default bucket: sagemaker-us-east-1-581187100103


In [54]:
# Define all pipeline parameters

DEFAULT_INPUT_DATA = f"s3://sagemaker-us-east-1-581187100103/umass/customer-churn/data/raw/storedata_total.csv"
DEFAULT_TARGET_COLUMN = "retained"

pipeline_name = "UMassChurnPipeline"
model_package_group_name = "UMassChurnModelPackageGroup"
base_job_prefix = "umass-churn"

processing_instance_type = ParameterString("ProcessingInstanceType", default_value="ml.m5.xlarge")
processing_instance_count = ParameterInteger("ProcessingInstanceCount", default_value=1)
training_instance_type = ParameterString("TrainingInstanceType", default_value="ml.m5.xlarge")
input_data = ParameterString("InputData", default_value=DEFAULT_INPUT_DATA)
target_column = ParameterString("TargetColumn", default_value=DEFAULT_TARGET_COLUMN)
test_size = ParameterFloat("TestSize", default_value=0.20)
auc_threshold = ParameterFloat("AucThreshold", default_value=0.70)
model_approval_status = ParameterString("ModelApprovalStatus", default_value="PendingManualApproval")

cache_config = CacheConfig(enable_caching=True, expire_after="30d")

print("Default input:", input_data.default_value)
print("Target:", target_column.default_value)

Default input: s3://sagemaker-us-east-1-581187100103/umass/customer-churn/data/raw/storedata_total.csv
Target: retained


In [55]:
# Create the preprocessing program

from pathlib import Path

PREPROCESS_SCRIPT = Path("../src/preprocess.py").resolve()

if not PREPROCESS_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Preprocessing script was not found: {PREPROCESS_SCRIPT}"
    )

print(f"Using preprocessing script: {PREPROCESS_SCRIPT}")

Using preprocessing script: /home/sagemaker-user/customer-churn-sagemaker/src/preprocess.py


In [56]:
# Define step_process

SAGEMAKER_SUPPRESS_V2_WARNING = 1 

sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type=processing_instance_type,
    instance_count=processing_instance_count,
    base_job_name=f"{base_job_prefix}-preprocessing",
    sagemaker_session=pipeline_session,
)

step_process_args = sklearn_processor.run(
    code=str(PREPROCESS_SCRIPT),
    inputs=[
        ProcessingInput(
            source=input_data,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/train"
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/validation"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/test"
        )
    ],
    arguments=[
        "--input-path",
        "/opt/ml/processing/input/storedata_total.csv",
        "--random-state",
        "42",
    ],
)

step_process = ProcessingStep(
    name="PreprocessChurnData",
    step_args=step_process_args,
    cache_config=cache_config,
)

print("Defined:", step_process.name)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


Defined: PreprocessChurnData


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [57]:
# Define the XGBoost estimator and step_tuning

MODEL_OUTPUT_PREFIX = "umass-churn/model-artifacts"
MODEL_OUTPUT_PATH = f"s3://{default_bucket}/{MODEL_OUTPUT_PREFIX}"

xgb_image_uri = image_uris.retrieve(
    framework="xgboost", region=region, version="1.7-1", py_version="py3", instance_type=training_instance_type.default_value
)

xgb_estimator = Estimator(
    image_uri=xgb_image_uri,
    role=role,
    instance_count=1,
    instance_type=training_instance_type,
    volume_size=20,
    max_run=3600,
    # output_path=f"s3://{default_bucket}/{base_job_prefix}/model-artifacts",
    output_path=MODEL_OUTPUT_PATH,
    base_job_name=f"{base_job_prefix}-xgb",
    sagemaker_session=pipeline_session,
)
xgb_estimator.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    num_round=300,
    early_stopping_rounds=20,
    tree_method="hist",
)

tuner = HyperparameterTuner(
    estimator=xgb_estimator,
    objective_metric_name="validation:auc",
    objective_type="Maximize",
    hyperparameter_ranges={
        "eta": ContinuousParameter(0.01, 0.30, scaling_type="Logarithmic"),
        "max_depth": IntegerParameter(3, 10),
        "min_child_weight": ContinuousParameter(1, 10),
        "subsample": ContinuousParameter(0.60, 1.00),
        "colsample_bytree": ContinuousParameter(0.60, 1.00),
        "gamma": ContinuousParameter(0, 5),
    },
    max_jobs=8,
    max_parallel_jobs=2,
    strategy="Bayesian",
)

step_tuning_args = tuner.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            content_type="text/csv",
        ),
    }
)
step_tuning = TuningStep(
    name="TuneChurnXGBoost",
    step_args=step_tuning_args,
    cache_config=cache_config,
)
print("Defined:", step_tuning.name)

INFO:sagemaker.image_uris:Ignoring unnecessary Python version: py3.


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: ml.m5.xlarge.


Defined: TuneChurnXGBoost


In [58]:
# Use the existing evaluation script

from pathlib import Path

EVALUATION_SCRIPT = Path("../src/evaluate.py").resolve()

if not EVALUATION_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Evaluation script was not found: {EVALUATION_SCRIPT}"
    )

print(f"Using evaluation script: {EVALUATION_SCRIPT}")

Using evaluation script: /home/sagemaker-user/customer-churn-sagemaker/src/evaluate.py


In [59]:
# Define evaluation processing step

from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.workflow.properties import PropertyFile

# This must match evaluate.py's output location:
# /opt/ml/processing/evaluation/evaluation.json
evaluation_report = PropertyFile(
    name="ChurnEvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

# evaluation_processor = SKLearnProcessor(
#     framework_version="1.2-1",
#     role=role,
#     instance_type=processing_instance_type,
#     instance_count=1,
#     base_job_name="umass-churn-evaluation",
#     sagemaker_session=pipeline_session,
# )

evaluation_processor = ScriptProcessor(
    image_uri=xgb_image_uri,
    command=["python3"],
    role=role,
    instance_type=processing_instance_type,
    instance_count=processing_instance_count,
    base_job_name="umass-churn-evaluation",
    sagemaker_session=pipeline_session,
)

MODEL_OUTPUT_PREFIX = "umass-churn/model-artifacts"

best_model_s3_uri = step_tuning.get_top_model_s3_uri(
    top_k=0,
    s3_bucket=default_bucket,
    prefix=MODEL_OUTPUT_PREFIX,
)

evaluation_args = evaluation_processor.run(
    code=str(EVALUATION_SCRIPT),
    inputs=[
        ProcessingInput(
            source=best_model_s3_uri,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_process.properties
                .ProcessingOutputConfig
                .Outputs["test"]
                .S3Output
                .S3Uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation",
        )
    ],
)

step_evaluate = ProcessingStep(
    name="EvaluateBestTunedModel",
    step_args=evaluation_args,
    property_files=[evaluation_report],  # This was missing
)

print("Evaluation step:", step_evaluate.name)
print("Property file:", evaluation_report.name)

Evaluation step: EvaluateBestTunedModel
Property file: ChurnEvaluationReport


In [60]:
# Define model registration and the AUC condition

from sagemaker.model import Model
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet

model = Model(
    image_uri=xgb_image_uri,
    model_data=best_model_s3_uri,
    role=role,
    sagemaker_session=pipeline_session,
)

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=[
        "ml.m5.large",
        "ml.m5.xlarge",
    ],
    transform_instances=["ml.m5.xlarge"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
)

step_register = ModelStep(
    name="RegisterChurnModel",
    step_args=register_args,
)

condition_auc = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_evaluate.name,
        property_file=evaluation_report,
        json_path="classification_metrics.auc.value",
    ),
    right=auc_threshold,
)

step_condition = ConditionStep(
    name="CheckChurnAUC",
    conditions=[condition_auc],
    if_steps=[step_register],
    else_steps=[],
)

print("Registration step:", step_register.name)
print("Condition step:", step_condition.name)
print("Required AUC:", auc_threshold.default_value)

Registration step: RegisterChurnModel
Condition step: CheckChurnAUC
Required AUC: 0.7


In [61]:
#  Construct, validate, and upsert the pipeline

from sagemaker.workflow.pipeline import Pipeline

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        input_data,
        processing_instance_type,
        processing_instance_count,
        training_instance_type,
        model_approval_status,
        auc_threshold,
    ],
    steps=[
        step_process,
        step_tuning,
        step_evaluate,
        step_condition,
    ],
    sagemaker_session=pipeline_session,
)

# Local structural validation: this catches undefined steps and many bad property references.
import json

definition = json.loads(pipeline.definition())

evaluation_step_definition = next(
    step
    for step in definition["Steps"]
    if step["Name"] == "EvaluateBestTunedModel"
)

print(
    json.dumps(
        evaluation_step_definition.get("PropertyFiles"),
        indent=2,
    )
)


[
  {
    "PropertyFileName": "ChurnEvaluationReport",
    "OutputName": "evaluation",
    "FilePath": "evaluation.json"
  }
]


In [62]:
# upsert the pipeline

upsert_response = pipeline.upsert(role_arn=role)
print(json.dumps(upsert_response, indent=2, default=str))


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


{
  "PipelineArn": "arn:aws:sagemaker:us-east-1:581187100103:pipeline/UMassChurnPipeline",
  "PipelineVersionId": 4,
  "ResponseMetadata": {
    "RequestId": "0174e486-3daf-422c-a60a-2720069a5ce8",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "0174e486-3daf-422c-a60a-2720069a5ce8",
      "strict-transport-security": "max-age=47304000; includeSubDomains",
      "x-frame-options": "DENY",
      "content-security-policy": "frame-ancestors 'none'",
      "cache-control": "no-cache, no-store, must-revalidate",
      "x-content-type-options": "nosniff",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "108",
      "date": "Sun, 26 Jul 2026 06:39:45 GMT"
    },
    "RetryAttempts": 0
  }
}


In [63]:
# Start the pipeline execution

execution = pipeline.start(
    parameters={
        "InputData": DEFAULT_INPUT_DATA,
        "ProcessingInstanceType": "ml.m5.xlarge",
        "ProcessingInstanceCount": 1,
        "TrainingInstanceType": "ml.m5.xlarge",
        "AucThreshold": 0.70,
        "ModelApprovalStatus": "PendingManualApproval",
    }
)
print("Execution ARN:", execution.arn)
print("Open SageMaker Studio > Pipelines >", pipeline_name, "to watch the graph and logs.")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Execution ARN: arn:aws:sagemaker:us-east-1:581187100103:pipeline/UMassChurnPipeline/execution/acv4p0wiy006
Open SageMaker Studio > Pipelines > UMassChurnPipeline to watch the graph and logs.


In [64]:
# Monitor the execution and inspect any failed step

execution.wait(delay=60, max_attempts=120)
print(json.dumps(execution.describe(), indent=2, default=str))

steps = execution.list_steps()
for item in steps:
    metadata = item.get("Metadata", {})
    print(item.get("StepName"), item.get("StepStatus"), item.get("FailureReason", ""), metadata)


{
  "PipelineArn": "arn:aws:sagemaker:us-east-1:581187100103:pipeline/UMassChurnPipeline",
  "PipelineExecutionArn": "arn:aws:sagemaker:us-east-1:581187100103:pipeline/UMassChurnPipeline/execution/acv4p0wiy006",
  "PipelineExecutionDisplayName": "execution-1785047991047",
  "PipelineExecutionStatus": "Succeeded",
  "PipelineExperimentConfig": {
    "ExperimentName": "UMassChurnPipeline",
    "TrialName": "acv4p0wiy006"
  },
  "CreationTime": "2026-07-26 06:39:50.979000+00:00",
  "LastModifiedTime": "2026-07-26 06:42:31.558000+00:00",
  "CreatedBy": {
    "UserProfileArn": "arn:aws:sagemaker:us-east-1:581187100103:user-profile/d-2ctlo94o5zbp/default-20260723T215583",
    "UserProfileName": "default-20260723T215583",
    "DomainId": "d-2ctlo94o5zbp",
    "IamIdentity": {
      "Arn": "arn:aws:sts::581187100103:assumed-role/AmazonSageMaker-ExecutionRole-20260723T215584/SageMaker",
      "PrincipalId": "AROAYOULTLXDZTUHPLQDQ:SageMaker"
    }
  },
  "LastModifiedBy": {
    "UserProfileArn":

In [65]:
# Inspect the registered model

import boto3
from pprint import pprint

sm_client = boto3.client("sagemaker", region_name="us-east-1")

model_package_arn = (
    "arn:aws:sagemaker:us-east-1:581187100103:"
    "model-package/UMassChurnModelPackageGroup/1"
)

model_details = sm_client.describe_model_package(
    ModelPackageName=model_package_arn
)

print("Model version:", model_details["ModelPackageVersion"])
print("Package status:", model_details["ModelPackageStatus"])
print("Approval status:", model_details["ModelApprovalStatus"])
print(
    "Model artifact:",
    model_details["InferenceSpecification"]["Containers"][0].get(
        "ModelDataUrl"
    )
)

Model version: 1
Package status: Completed
Approval status: PendingManualApproval
Model artifact: s3://sagemaker-us-east-1-581187100103/umass-churn/model-artifacts/0r4s6j6vd2xg-TuneChur-beQDkrMceV-007-3de5ef82/output/model.tar.gz


In [67]:
# Verify status

model_details = sm_client.describe_model_package(
    ModelPackageName=model_package_arn
)

print(model_details["ModelApprovalStatus"])

Approved
